# MongoDB Integration

In this notebook, the Olist e-commerce data is loaded, merged into a simplified structure, stored in MongoDB, indexed, and queried for analysis.  
The goal is to demonstrate how MongoDB can be used to store and analyze processed e-commerce data efficiently.

# 01 Import Libraries

In this section, we import the Python libraries needed for MongoDB connection, 
data handling, and performance testing.

In [1]:
from pymongo import MongoClient
import pandas as pd
import time

# 02 Connect to MongoDB

Here, we create a connection to the local MongoDB server and define the database 
and collection that will be used in this project.

In [2]:
client = MongoClient("mongodb://localhost:27017/")
db = client["olist_ecommerce"]
collection = db["orders_simple"]

print("Connected to MongoDB")

Connected to MongoDB


# 03 Load Data

In this section, we load the required CSV files containing orders, customers, and payment data.  
These datasets are later merged so that the most important information can be stored in MongoDB in a simple and query-friendly format.

In [3]:
import pandas as pd
import os

print("Current working directory:", os.getcwd())

orders_path = "../data/raw/olist_orders_dataset.csv"
customers_path = "../data/raw/olist_customers_dataset.csv"
payments_path = "../data/raw/olist_order_payments_dataset.csv"

print("Orders exists:", os.path.exists(orders_path))
print("Customers exists:", os.path.exists(customers_path))
print("Payments exists:", os.path.exists(payments_path))

orders = pd.read_csv(orders_path)
customers = pd.read_csv(customers_path)
payments = pd.read_csv(payments_path)

print("Data loaded successfully")
print("orders shape:", orders.shape)
print("customers shape:", customers.shape)
print("payments shape:", payments.shape)

Current working directory: c:\Users\John\Desktop\data_sience_group\BigData-Ecommerce-Analysis\notebooks
Orders exists: True
Customers exists: True
Payments exists: True
Data loaded successfully
orders shape: (99441, 8)
customers shape: (99441, 5)
payments shape: (103886, 5)


# 04 Merge Datasets

Here, we combine the orders, customers, and payments datasets into one merged dataset.  
This makes the data easier to work with in MongoDB, because each document will contain the key information needed for analysis.

In [4]:
payments_sum = payments.groupby("order_id", as_index=False)["payment_value"].sum()

merged = orders.merge(customers, on="customer_id", how="left")
merged = merged.merge(payments_sum, on="order_id", how="left")

merged["payment_value"] = merged["payment_value"].fillna(0)
merged = merged.where(pd.notnull(merged), None)

merged.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62


# 05 Insert into MongoDB

In this step, the merged pandas DataFrame is converted into JSON-like documents and inserted into the MongoDB collection.  
Each document represents one order enriched with customer and payment information.

In [5]:
collection.delete_many({})

docs = merged.to_dict("records")
collection.insert_many(docs)

print("Inserted documents:", collection.count_documents({}))

Inserted documents: 99441


# 06 Create Indexes

Indexes are created to improve query performance.  
The selected fields are commonly used in filtering and lookup operations, such as order ID, customer state, and order status.

In [6]:
collection.create_index("order_id", unique=True)
collection.create_index("customer_state")
collection.create_index("order_status")

'order_status_1'

# 07 Run Queries

This section contains the main MongoDB queries used in the project.  
The queries are used to count documents, filter orders by state, and aggregate sales by state.

### Query 1: Total number of orders

In [7]:
print("Total orders:", collection.count_documents({}))

Total orders: 99441


### Query 2: Delivered orders

In [8]:
print("Delivered orders:", collection.count_documents({"order_status": "delivered"}))

Delivered orders: 96478


### Query 3: Orders from SP

In [9]:
results = list(collection.find(
    {"customer_state": "SP"},
    {"_id": 0, "order_id": 1, "customer_city": 1, "payment_value": 1}
).limit(5))

results

[{'order_id': 'e481f51cbdc54678b7cc49136f2d6af7',
  'customer_city': 'sao paulo',
  'payment_value': 38.71},
 {'order_id': 'ad21c59c0840e6cb83a9ceb5573f8159',
  'customer_city': 'santo andre',
  'payment_value': 28.62},
 {'order_id': 'e69bfb5eb88e0ed6a785585b27e16dbf',
  'customer_city': 'sorocaba',
  'payment_value': 169.76},
 {'order_id': '34513ce0c4fab462a55830c0989c7edb',
  'customer_city': 'sao paulo',
  'payment_value': 114.13},
 {'order_id': '5ff96c15d0b717ac6ad1f3d77225a350',
  'customer_city': 'sao paulo',
  'payment_value': 32.7}]

### Query 4: Sales by state

In [10]:
pipeline = [
    {
        "$group": {
            "_id": "$customer_state",
            "orders": {"$sum": 1},
            "total_sales": {"$sum": "$payment_value"}
        }
    },
    {"$sort": {"total_sales": -1}}
]

list(collection.aggregate(pipeline))[:10]

[{'_id': 'SP', 'orders': 41746, 'total_sales': 5998226.96},
 {'_id': 'RJ', 'orders': 12852, 'total_sales': 2144379.69},
 {'_id': 'MG', 'orders': 11635, 'total_sales': 1872257.26},
 {'_id': 'RS', 'orders': 5466, 'total_sales': 890898.54},
 {'_id': 'PR', 'orders': 5045, 'total_sales': 811156.38},
 {'_id': 'SC', 'orders': 3637, 'total_sales': 623086.43},
 {'_id': 'BA', 'orders': 3380, 'total_sales': 616645.82},
 {'_id': 'DF', 'orders': 2140, 'total_sales': 355141.08},
 {'_id': 'GO', 'orders': 2020, 'total_sales': 350092.31},
 {'_id': 'ES', 'orders': 2033, 'total_sales': 325967.55}]

# 08 Performance Test

In this section, we measure the execution time of a filtered MongoDB query.  
This gives a simple example of query performance after indexing.

In [11]:
start = time.time()
list(collection.find({"customer_state": "SP"}))
end = time.time()

print("Query time:", end - start)

Query time: 0.13524436950683594


# Conclusion

In this notebook, the Olist orders, customers, and payments datasets were merged and stored in MongoDB.  
Indexes were added to improve query performance, and several queries were used to analyze the data, including total orders, delivered orders, filtered orders from São Paulo, and sales by state.